# Docker, Dockerfile, Docker Compose & Multi-Stage Build (Enterprise AI Deployment)

This is one of the most important topics for **Senior AI Engineer**, **GenAI Engineer**, and **Agentic AI Engineer** interviews.

The interviewer usually asks:

> **"You have built a FastAPI + LangGraph + Bedrock application. How will you deploy it?"**

A good answer always starts with **Docker**.

---

# 1. What is Docker?

## Definition

Docker is a containerization platform that packages an application along with its runtime, libraries, and dependencies into a **container**, ensuring it runs consistently across development, testing, and production.

---

## Interview Answer

> Docker packages an application with all its dependencies into a lightweight container so that it runs consistently across different environments.

---

# Why Docker?

Without Docker

```text
Developer Laptop

Python 3.12

↓

Works
```

Production

```text
Python 3.10

↓

Different Library Version

↓

Application Crash
```

---

With Docker

```text
Application

+

Python

+

Libraries

+

OS Dependencies

↓

Docker Image

↓

Runs Anywhere
```

---

# Benefits

- Same environment everywhere
- Easy deployment
- Easy scaling
- Lightweight
- Fast startup
- Portable

---

# 2. Docker Architecture

```text
                    Developer

                        │

                 docker build

                        │

                        ▼

                  Docker Image

                        │

                  docker run

                        │

                        ▼

                 Docker Container
```

---

# Docker Components

| Component | Purpose |
|------------|----------|
| Dockerfile | Build instructions |
| Image | Application template |
| Container | Running application |
| Registry | Stores images (Docker Hub/ECR/ACR) |

---

# Image vs Container

Image

```text
Blueprint
```

Container

```text
Running Application
```

Example

```text
Image

↓

Container 1

Container 2

Container 3
```

One image can create many containers.

---

# 3. Enterprise AI Deployment

AWS

```text
Docker

↓

Amazon ECR

↓

Amazon ECS Fargate

↓

Running AI Application
```

Azure

```text
Docker

↓

Azure Container Registry

↓

Azure Container Apps

↓

Running AI Application
```

---

# 4. Docker Workflow

```text
Write Code

↓

Dockerfile

↓

Build Image

↓

Run Container

↓

Push Image

↓

Deploy
```

---

# 5. Dockerfile

A Dockerfile contains instructions for building a Docker image.

---

## Sample Project

```text
hr-ai/

│── app.py

│── requirements.txt

│── Dockerfile

│── docker-compose.yml
```

---

# Dockerfile

```dockerfile
# ==========================================================
# Base Image
# Purpose:
# Use official Python image.
# ==========================================================

FROM python:3.12-slim


# ==========================================================
# Working Directory
# Purpose:
# All commands execute inside /app
# ==========================================================

WORKDIR /app


# ==========================================================
# Copy Requirements
# Purpose:
# Copy only requirements first to improve Docker caching.
# ==========================================================

COPY requirements.txt .


# ==========================================================
# Install Dependencies
# Purpose:
# Install Python packages.
# ==========================================================

RUN pip install --no-cache-dir -r requirements.txt


# ==========================================================
# Copy Application
# Purpose:
# Copy remaining project files.
# ==========================================================

COPY . .


# ==========================================================
# Expose Port
# Purpose:
# FastAPI runs on port 8000.
# ==========================================================

EXPOSE 8000


# ==========================================================
# Start Application
# Purpose:
# Launch FastAPI using Uvicorn.
# ==========================================================

CMD ["uvicorn","app:app","--host","0.0.0.0","--port","8000"]
```

---

# Dockerfile Instructions

| Instruction | Purpose |
|------------|----------|
| FROM | Base image |
| WORKDIR | Working directory |
| COPY | Copy files |
| RUN | Execute commands during build |
| ENV | Environment variables |
| EXPOSE | Open container port |
| CMD | Default startup command |
| ENTRYPOINT | Fixed executable |

---

# Build Docker Image

```bash
docker build -t hr-ai .
```

Meaning

```text
docker build

↓

Create Image

↓

Image Name = hr-ai
```

---

# List Images

```bash
docker images
```

---

# Run Container

```bash
docker run -p 8000:8000 hr-ai
```

Meaning

```text
Host Port 8000

↓

Container Port 8000
```

---

# Running Containers

```bash
docker ps
```

---

# Stop Container

```bash
docker stop <container_id>
```

---

# Remove Container

```bash
docker rm <container_id>
```

---

# Remove Image

```bash
docker rmi hr-ai
```

---

# 6. Docker Compose

## What is Docker Compose?

Docker Compose runs **multiple containers together**.

Instead of

```text
FastAPI

Redis

PostgreSQL

Qdrant

Run separately
```

Docker Compose starts everything using one file.

---

# Architecture

```text
                Docker Compose

                     │

     ┌───────────────┼──────────────┐

     ▼               ▼              ▼

 FastAPI         PostgreSQL      Redis

                     ▼

                  Qdrant
```

---

# docker-compose.yml

```yaml
version: "3.9"

services:

  api:

    build: .

    ports:
      - "8000:8000"

    depends_on:
      - postgres
      - redis
      - qdrant

  postgres:

    image: postgres:16

    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: password
      POSTGRES_DB: hr_ai

    ports:
      - "5432:5432"

  redis:

    image: redis:7

    ports:
      - "6379:6379"

  qdrant:

    image: qdrant/qdrant

    ports:
      - "6333:6333"
```

---

# Run Everything

```bash
docker compose up
```

---

Run in Background

```bash
docker compose up -d
```

---

Stop

```bash
docker compose down
```

---

# Why Docker Compose?

Perfect for

- Local Development
- Integration Testing
- Running AI Stack locally

---

# 7. Multi-Stage Build

## Problem

Normal Image

```text
Python

↓

Build Tools

↓

Compiler

↓

Application

↓

2 GB Image
```

Large image.

---

## Solution

Multi-stage build.

Build in one stage.

Run in another.

---

# Multi-stage Dockerfile

```dockerfile
# ==========================================================
# Stage 1 : Build
# ==========================================================

FROM python:3.12-slim AS builder

WORKDIR /app

COPY requirements.txt .

RUN pip install --prefix=/install --no-cache-dir -r requirements.txt

COPY . .


# ==========================================================
# Stage 2 : Runtime
# ==========================================================

FROM python:3.12-slim

WORKDIR /app

COPY --from=builder /install /usr/local

COPY . .

EXPOSE 8000

CMD ["uvicorn","app:app","--host","0.0.0.0","--port","8000"]
```

---

# Why Multi-stage Build?

Without

```text
Image

↓

2 GB
```

With

```text
Image

↓

300 MB
```

Benefits

- Smaller image
- Faster deployment
- Better security
- Faster startup

---

# Production Deployment

AWS

```text
GitHub

↓

GitHub Actions

↓

Docker Build

↓

Amazon ECR

↓

Amazon ECS Fargate

↓

Application Load Balancer

↓

API Gateway

↓

Users
```

Azure

```text
GitHub

↓

GitHub Actions

↓

Azure Container Registry

↓

Azure Container Apps

↓

Application Gateway

↓

API Management

↓

Users
```

---

# Best Practices

✅ Use `python:3.x-slim`

✅ Use `.dockerignore`

✅ Install only required packages

✅ Never store secrets inside the image

✅ Use environment variables

✅ Health check endpoints

✅ Use Multi-stage Build

✅ Pin dependency versions

---

# Common Interview Questions

### Q1. Why Docker?

To package the application with all dependencies and ensure consistent execution.

---

### Q2. Docker vs Virtual Machine?

| Docker | VM |
|---------|----|
| Lightweight | Heavy |
| Shares host kernel | Own OS |
| Fast startup | Slower startup |
| Less memory | More memory |

---

### Q3. Why Docker Compose?

To run multiple related containers such as FastAPI, PostgreSQL, Redis, and Qdrant together.

---

### Q4. Why Multi-stage Build?

To reduce image size by separating the build environment from the runtime environment.

---

### Q5. Why copy `requirements.txt` first?

Docker caches layers. If only application code changes and `requirements.txt` remains the same, dependencies are not reinstalled, making builds much faster.

---

### Q6. Why use `python:3.12-slim`?

It provides a smaller image than the full Python image, reducing download time, storage, and attack surface.

---

### Q7. Why expose port 8000?

Uvicorn, which serves FastAPI, listens on port **8000** by default.

---

# EPAM Senior Answer (3 Minutes)

> "For an enterprise AI application, I first containerize the FastAPI and LangGraph application using Docker. The Dockerfile uses a lightweight Python base image, installs dependencies, copies the application, exposes port 8000, and starts the service with Uvicorn. For local development, I use Docker Compose to run FastAPI together with PostgreSQL, Redis, and Qdrant in a single command. For production, I build a multi-stage Docker image to reduce image size and improve security by excluding unnecessary build tools from the final runtime image. The image is pushed to Amazon ECR on AWS or Azure Container Registry on Azure, and deployed to Amazon ECS Fargate or Azure Container Apps. Traffic is routed through an Application Load Balancer or Azure Application Gateway and exposed using API Gateway or API Management. Secrets are managed using AWS Secrets Manager or Azure Key Vault, and the application is monitored using CloudWatch or Azure Monitor with LangSmith for AI workflow observability. This approach provides a scalable, secure, and production-ready deployment pipeline."